In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta
from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch

symbol = 'BTCUSDT'
market_type = 'SPOT'
discretization = '1D'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 17
macd_buffer = 200
atr_buffer = 200
display_start_date_str = '2024-01-01' # дата, с которой вы хотите отображать '2026-03-25 14:30:00' -- формат даты и времени
display_start_dt = parser.parse(display_start_date_str)

# Определяем таймфрейм в секундах
discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

# Рассчитываем дату начала загрузки с буфером
load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

print(f"Загружаем данные с: {load_start_dt} (буфер {mrc_buffer} свечей)")
print(f"Отображаем с: {display_start_date_str}")

mrc_df = fetch(market_type, symbol, discretization, load_start_dt)
df = mrc_df.iloc[mrc_buffer:].copy()

print(df.head())

In [ ]:
import ta
import numpy as np
import pandas as pd
#Process iteration over timeseries dataframe with sliding window approach
window_size = 30

def mrc_supersmoother(src, length):
    """
    Supersmoother filter by John Ehlers
    """
    a1 = np.exp(-1.414 * np.pi / length)
    b1 = 2 * a1 * np.cos(1.414 * np.pi / length)
    c3 = -a1 * a1
    c2 = b1
    c1 = 1 - c2 - c3

    ss = np.zeros_like(src)
    ss[:2] = src[:2]

    for i in range(2, len(src)):
        ss[i] = c1 * src[i] + c2 * ss[i-1] + c3 * ss[i-2]

    return ss

def mrc_sak_filter(filter_type, src, length):
    """
    Ehlers Swiss Army Knife filters
    """
    pi = np.pi
    cycle = 2 * pi / length

    c0, c1 = 1.0, 0.0
    b0, b1, b2 = 1.0, 0.0, 0.0
    a1, a2 = 0.0, 0.0
    alpha, beta, gamma = 0.0, 0.0, 0.0

    if filter_type == "Ehlers EMA":
        alpha = (np.cos(cycle) + np.sin(cycle) - 1) / np.cos(cycle)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "Gaussian":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "Butterworth":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha / 4
        b1, b2 = 2, 1
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "BandStop":
        beta = np.cos(cycle)
        gamma = 1 / np.cos(cycle * 2 * 0.1)  # delta = 0.1
        alpha = gamma - np.sqrt(gamma * gamma - 1)
        c0 = (1 + alpha) / 2
        b1 = -2 * beta
        b2 = 1
        a1 = beta * (1 + alpha)
        a2 = -alpha

    elif filter_type == "SMA":
        c1 = 1 / length
        b0 = 1 / length
        a1 = 1

    elif filter_type == "EMA":
        alpha = 2 / (length + 1)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "RMA":
        alpha = 1 / length
        b0 = alpha
        a1 = 1 - alpha

    output = np.zeros_like(src)
    output[:3] = src[:3]

    for i in range(3, len(src)):
        output[i] = (c0 * (b0 * src[i] +
                          b1 * (src[i-1] if i-1 >= 0 else 0) +
                          b2 * (src[i-2] if i-2 >= 0 else 0)) +
                    a1 * output[i-1] +
                    a2 * output[i-2] -
                    c1 * (src[i-length] if i-length >= 0 else 0))

    return output

def mrc_calculate(source_type='hlc3', filter_type='SuperSmoother', innermult=1.0, outermult=2.415):
    """
    Calculate Mean Reversion Channel

    Использует глобальные переменные:
    - mrc_df: DataFrame с данными (включая буфер)
    - df: DataFrame для отображения (без буфера)
    """
    global mrc_df, df

    # Calculate source price на mrc_df (с буфером)
    if source_type == 'hlc3':
        source = (mrc_df['high'] + mrc_df['low'] + mrc_df['close']) / 3
    elif source_type == 'ohlc4':
        source = (mrc_df['open'] + mrc_df['high'] + mrc_df['low'] + mrc_df['close']) / 4
    else:  # 'close'
        source = mrc_df['close']

    source = source.values

    # Calculate True Range на mrc_df
    tr = np.maximum(
        mrc_df['high'] - mrc_df['low'],
        np.maximum(
            abs(mrc_df['high'] - mrc_df['close'].shift(1)),
            abs(mrc_df['low'] - mrc_df['close'].shift(1))
        )
    ).fillna(0).values

    length = 200

    # Apply filters
    if filter_type == 'SuperSmoother':
        meanline = mrc_supersmoother(source, length)
        meanrange = mrc_supersmoother(tr, length)
    else:
        meanline = mrc_sak_filter(filter_type, source, length)
        meanrange = mrc_sak_filter(filter_type, tr, length)

    pi = np.pi
    mult = pi * innermult
    mult2 = pi * outermult

    # Calculate channels
    upband1 = meanline + (meanrange * mult)
    loband1 = meanline - (meanrange * mult)
    upband2 = meanline + (meanrange * mult2)
    loband2 = meanline - (meanrange * mult2)

    # Сохраняем результаты в mrc_df
    mrc_df['meanline'] = meanline
    mrc_df['meanrange'] = meanrange
    mrc_df['upband1'] = upband1
    mrc_df['loband1'] = loband1
    mrc_df['upband2'] = upband2
    mrc_df['loband2'] = loband2

    # Переносим результаты из mrc_df в df (видимый диапазон)
    df['meanline'] = mrc_df.loc[df.index, 'meanline']
    df['meanrange'] = mrc_df.loc[df.index, 'meanrange']
    df['upband1'] = mrc_df.loc[df.index, 'upband1']
    df['loband1'] = mrc_df.loc[df.index, 'loband1']
    df['upband2'] = mrc_df.loc[df.index, 'upband2']
    df['loband2'] = mrc_df.loc[df.index, 'loband2']

def stochastic_tradingview(high, low, close, periodK=14, smoothK=3, periodD=3):
    lowest_low = low.rolling(window=periodK).min()
    highest_high = high.rolling(window=periodK).max()
    raw_k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    stoch_k = raw_k.rolling(window=smoothK).mean()
    stoch_d = stoch_k.rolling(window=periodD).mean()

    return stoch_k, stoch_d

def find_target_candles(df, window=10):
    """
    Поиск целевых свечей по критериям:
    1. Касание линии R2 (upband2) или S2 (loband2) тенью или телом
       - Свеча пересекает линию (часть выше, часть ниже)
       - ИЛИ свеча полностью выше R2
       - ИЛИ свеча полностью ниже S2
    2. Объем выше среднего за 10 предыдущих свечей
    3. Тень больше средней тени за 10 предыдущих свечей
    """
    # Расчет среднего объема за 10 предыдущих свечей
    df['volume_ma'] = df['volume'].rolling(window=window, min_periods=1).mean()

    # Расчет теней
    df['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
    df['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']
    df['max_shadow'] = df[['upper_shadow', 'lower_shadow']].max(axis=1)

    # Средняя тень за 10 предыдущих свечей (исключая текущую)
    df['shadow_ma'] = df['max_shadow'].shift(1).rolling(window=window, min_periods=1).mean()

    # Поиск целевых свечей
    target_conditions = []

    for i in range(len(df)):
        # Пропускаем первые 10 свечей (недостаточно данных для сравнения)
        if i < window:
            target_conditions.append(False)
            continue

        # 1. Проверка касания с R2 или S2
        high = df['high'].iloc[i]
        low = df['low'].iloc[i]
        r2 = df['upband2'].iloc[i]
        s2 = df['loband2'].iloc[i]

        # Варианты касания с R2:
        # - Пересечение: high >= r2 и low <= r2
        # - Полностью выше: low > r2
        touches_r2 = (high >= r2 and low <= r2) or (low > r2)

        # Варианты касания с S2:
        # - Пересечение: high >= s2 и low <= s2
        # - Полностью ниже: high < s2
        touches_s2 = (high >= s2 and low <= s2) or (high < s2)

        # Касание хотя бы одной линии
        touches_level = touches_r2 or touches_s2

        # 2. Объем выше среднего за предыдущие 10 свечей
        volume_above = df['volume'].iloc[i] > df['volume_ma'].iloc[i-1] if i > 0 else False

        # 3. Тень больше средней за предыдущие 10 свечей
        shadow_above = df['max_shadow'].iloc[i] > df['shadow_ma'].iloc[i]

        # Все условия вместе
        is_target = touches_level and volume_above and shadow_above
        target_conditions.append(is_target)

    df['is_target_candle'] = target_conditions

    return df

def get_target_candle_type(row):
    """Определяет тип целевой свечи"""
    if row['is_target_candle']:
        if (row['high'] >= row['upband2'] and row['low'] <= row['upband2']) or (row['low'] > row['upband2']):
            return 'r2'
        if (row['high'] >= row['loband2'] and row['low'] <= row['loband2']) or (row['high'] < row['loband2']):
            return 's2'

    return None




# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# Функция для подготовки данных с буфером
def prepare_buffer_data(df_main, df_display, buffer_size):
    combined = pd.concat([df_main.iloc[-(buffer_size + len(df_display)):], df_display])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    return combined

# 1. RSI
rsi_calc_df = prepare_buffer_data(mrc_df, df, rsi_buffer)
df['rsi'] = ta.momentum.RSIIndicator(close=rsi_calc_df['close'], window=14).rsi().loc[df.index]

# 2. Stochastic
stoch_calc_df = prepare_buffer_data(mrc_df, df, stoch_buffer)
stoch_k_full, stoch_d_full = stochastic_tradingview(
    high=stoch_calc_df['high'],
    low=stoch_calc_df['low'],
    close=stoch_calc_df['close']
)
df['stoch_k'] = stoch_k_full.loc[df.index]
df['stoch_d'] = stoch_d_full.loc[df.index]

# 3. MACD
macd_calc_df = prepare_buffer_data(mrc_df, df, macd_buffer)
macd = ta.trend.MACD(
    close=macd_calc_df['close'],
    window_slow=26,
    window_fast=12,
    window_sign=9
)
df['macd_line'] = macd.macd().loc[df.index]
df['macd_signal'] = macd.macd_signal().loc[df.index]
df['macd_histogram'] = macd.macd_diff().loc[df.index]

# 4. ATR
atr_calc_df = prepare_buffer_data(mrc_df, df, atr_buffer)
atr_full = ta.volatility.AverageTrueRange(
    high=atr_calc_df['high'],
    low=atr_calc_df['low'],
    close=atr_calc_df['close'],
    window=14
).average_true_range()
df['atr'] = atr_full.loc[df.index]

for i in range(len(df) - window_size + 1):
    window = df.iloc[i:i + window_size]
    # process window
    print(f"{window.index[0]} - {window.index[-1]}")

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots

is_log_scale_by_default=True
candlestick_row = 1
volume_row = 2
rsi_row = 3
stoch_row = 4
macd_row = 5
atr_row = 6

def add_bars(col, name, color, row):
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df[col],
            name=name,
            marker=dict(color=color),
            width=(df.index[1] - df.index[0]).total_seconds() * 1000,
        ),
        row=row, col=1
)

def add_scatter(col, name, color, row, fill=None, fillcolor=None, width=2, dash=None):
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df[col],
            name=name,
            line=dict(color=color, width=width, dash=dash),
            mode='lines',
            fill=fill,
            fillcolor=fillcolor
        ),
        row=row, col=1
    )

def add_target_candle_scatter(target_candle_type, position, position_multiplier, name, color, symbol_direction):
    signals = df[df['target_candle_type'] == target_candle_type]

    if len(signals) > 0:
        fig.add_trace(
            go.Scatter(
                x=signals.index,
                y=signals[position] * position_multiplier,
                name=name + " TC",
                mode='markers',
                marker=dict(color=color, size=10, symbol='triangle-' + symbol_direction)
            ),
            row=candlestick_row, col=1
        )

def add_over_zone(y0, y1, fillcolor, row):
    fig.add_hrect(
        y0=y0, y1=y1,
        line_width=0,
        fillcolor=fillcolor,
        opacity=0.2,
        row=row, col=1
    )

def add_central_line(row, y=50, line_dash="dot"):
    fig.add_hline(
        y=y,
        line_dash=line_dash,
        line_color="white",
        line_width=1,
        opacity=0.3,
        row=row, col=1
    )

def add_over_zones_and_a_central_line(row):
    add_over_zone(80, 100, "red", row)
    add_over_zone(0, 20, "green", row)
    add_central_line(row)

def add_stoch(speed, color):
    add_scatter('stoch_' + speed, "Stoch %" + speed.capitalize(), color, stoch_row)

fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    row_heights=[7, 1, 3, 3, 3, 2],
    vertical_spacing=0.03
)

fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

mrc_calculate()
add_scatter('meanline', "MRC Mean", '#FFCD00', candlestick_row)
add_scatter('upband1', "MRC R1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('loband1', "MRC S1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('upband2', "MRC R2", 'red', candlestick_row, width=1)
add_scatter('loband2', "MRC S2", 'red', candlestick_row, width=1)

add_bars(
    "volume",
    "Volume",
    ["green" if c > o else "red" for o, c in zip(df["open"], df["close"])],
    volume_row)

add_scatter('rsi', "RSI", 'purple', rsi_row)
add_over_zones_and_a_central_line(rsi_row)

add_stoch('k', "lightblue")
add_stoch('d', "orange")
add_over_zones_and_a_central_line(stoch_row)

full_histogram = macd.macd_diff()
prev_histogram = full_histogram.shift(1).loc[df.index]
conditions = [
    (df['macd_histogram'] >= 0) & (df['macd_histogram'] >= prev_histogram),
    (df['macd_histogram'] >= 0) & (df['macd_histogram'] < prev_histogram),
    (df['macd_histogram'] < 0) & (df['macd_histogram'] <= prev_histogram),
    (df['macd_histogram'] < 0) & (df['macd_histogram'] > prev_histogram)
]
choices = ['green', 'lightgreen', 'red', 'lightcoral']
macd_colors = np.select(conditions, choices, default='rgba(128, 128, 128, 0.3)')
add_scatter("macd_line", "MACD Line", 'lightblue', macd_row)
add_scatter("macd_signal", "Signal Line", 'orange', macd_row)
add_bars("macd_histogram", "MACD Histogram", macd_colors, macd_row)
add_central_line(macd_row, line_dash="solid")

add_scatter('atr', "ATR (14)", 'orange', atr_row, fill='tozeroy', fillcolor='rgba(255, 165, 0, 0.1)')
add_central_line(atr_row, y=df['atr'].mean())

# Применяем функцию
df = find_target_candles(df)
# Выводим статистику
target_candles_count = df['is_target_candle'].sum()
print(f"Найдено целевых свечей: {target_candles_count}")

if target_candles_count > 0:
    print("\nДаты целевых свечей:")
    target_candle_dates = df[df['is_target_candle']].index

    for date in target_candle_dates:
        print(f"  - {date}")

df['target_candle_type'] = df.apply(get_target_candle_type, axis=1)
add_target_candle_scatter('r2', 'high', 1.005, "🔴 Short", 'red', "down")
add_target_candle_scatter('s2', 'low', 0.995, "🟢 Long", 'green', "up")

# Разделяем по типам касания
cross_r2 = (df['high'] >= df['upband2']) & (df['low'] <= df['upband2'])
above_r2 = df['low'] > df['upband2']
cross_s2 = (df['high'] >= df['loband2']) & (df['low'] <= df['loband2'])
below_s2 = df['high'] < df['loband2']

df['touch_type'] = 'none'
df.loc[cross_r2, 'touch_type'] = 'cross_r2'
df.loc[above_r2, 'touch_type'] = 'above_r2'
df.loc[cross_s2, 'touch_type'] = 'cross_s2'
df.loc[below_s2, 'touch_type'] = 'below_s2'

# Просмотр найденных свечей с типом касания
target_candles_info = df[df['is_target_candle']][['open', 'high', 'low', 'close', 'volume', 'touch_type']]
print("\nДетали целевых свечей:")
print(target_candles_info)

price_log_scale_value="log"
price_linear_scale_value="linear"
price_log_scale_title="Price (log scale)"
price_linear_scale_title="Price (linear scale)"

fig.update_layout(
    title="OHLC with Volume Explosion After Valley",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    yaxis2_title="Volume",
    yaxis3_title="RSI",
    yaxis4_title="Stoch",
    yaxis5_title="MACD",
    yaxis6_title="ATR",
    height=3000,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.075,
            y=1.08,
            buttons=[
                dict(
                    label="Linear",
                    method="relayout",
                    args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                ),
                dict(
                    label="Log",
                    method="relayout",
                    args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                )
            ],
            font=dict(
                color="red",
                size=12,
                family="Arial"
            ),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

fig.show()